# Clean-only one-class detector on the shared split (Sandra)

CPU runtime is enough (about 1 s of feature extraction per adapter).
Uses `shared_protocol.detectors.oneclass`: Ledoit-Wolf Mahalanobis distance on 30 all-layer spectral statistics, fitted on the **benign** adapters of the shared train split only (backdoored train adapters are never opened). No hyper-parameter is tuned. Freeze, then calibrate on clean calibration adapters.
Evaluation stays **locked** until every detector is frozen.

In [ ]:
!pip -q install safetensors scikit-learn
from google.colab import drive
drive.mount('/content/drive')

REPO_URL   = "https://github.com/<ORG>/FYP-AI-Antivirus.git"
REPO_BRANCH = "sandra/peftguard-oneclass"
MANIFEST   = "/content/drive/MyDrive/fyp_shared/manifest.json"
SPLIT      = "/content/drive/MyDrive/fyp_shared/splits.json"
ADAPTER_ROOT = "/content/adapters"            # <root>/<adapter_id>/adapter_model.safetensors (same staging as the PEFTGuard notebook)
CACHE_DIR  = "/content/oneclass_cache"           # feature cache keyed by file SHA256 + extractor version
RUN_DIR    = "/content/runs/oneclass"
DRIVE_OUT  = "/content/drive/MyDrive/fyp_shared/oneclass"
SEED = 0

In [ ]:
import os, sys, shutil, re
from pathlib import Path
if not os.path.isdir("/content/repo"):
    !git clone --branch $REPO_BRANCH $REPO_URL /content/repo
%cd /content/repo
sys.path.insert(0, "/content/repo")
!git log -1 --format='%H %s'

from shared_protocol.manifest import load_manifest
from shared_protocol.splits import load_split, development_records
from shared_protocol.scoring import bind_protocol
manifest = load_manifest(MANIFEST, require_verified=True)
plan = load_split(SPLIT, manifest)
binding = bind_protocol(manifest, plan)
train_records = development_records(manifest, plan, "train")
val_records = development_records(manifest, plan, "validation")
cal_records = development_records(manifest, plan, "calibration")
print({p: len(binding.ids(p)) for p in ("train", "validation", "calibration", "evaluation", "final_holdout")})

Stage adapter files (skip the files already staged by the PEFTGuard notebook). Add `HF_TOKEN` in Colab Secrets for the private repo.

In [ ]:
from huggingface_hub import hf_hub_download
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

def stage(records):
    for row in records:
        dest = Path(ADAPTER_ROOT) / row["adapter_id"] / "adapter_model.safetensors"
        if dest.exists():
            continue
        m = re.fullmatch(r"hf://datasets/([^@]+)@([^/]+)/(.+)", row["path_or_source"])
        if not m:
            raise FileNotFoundError(f"{row['adapter_id']}: not an hf:// locator; stage it manually to {dest}")
        repo, rev, path = m.groups()
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(hf_hub_download(repo, path, repo_type="dataset", revision=rev, token=HF_TOKEN), dest)

stage(train_records); stage(val_records); stage(cal_records)

## Fit (benign train adapters only; every file is re-hashed against the manifest first)

In [ ]:
from shared_protocol.detectors.oneclass.train import OneClassConfig, fit
from shared_protocol.detectors.peftguard.converter import root_resolver
cfg = OneClassConfig()          # prefer >= 50 clean train adapters; min_clean=10 is the hard floor
result = fit(binding, train_records, val_records, cfg, RUN_DIR, resolver=root_resolver(ADAPTER_ROOT), cache_dir=CACHE_DIR)
print(result["n_fit"], "clean adapters | shrinkage", result["shrinkage"])
print(result["model_file_sha256"], "<- record this in the team log (the frozen state)")
os.makedirs(DRIVE_OUT, exist_ok=False)
shutil.copyfile(result["model_file"], f"{DRIVE_OUT}/model.json")

## Freeze: clean-only 5% conformal threshold (calibration partition only)

In [ ]:
from shared_protocol.detectors.oneclass.detector import OneClassDetector
from shared_protocol.scoring import score_development
from shared_protocol.calibration import calibrate_benign, save_threshold
detector = OneClassDetector.from_model(result["model_file"], binding, train_records, expected_file_sha256=result["model_file_sha256"],
                                    resolver=root_resolver(ADAPTER_ROOT), cache_dir=CACHE_DIR)
cal_scores = score_development(detector, cal_records, binding, "calibration")
threshold = calibrate_benign(binding, cal_records, cal_scores)
print("n calibration:", len(cal_scores.scores), "| threshold:", threshold.value)
save_threshold(f"{DRIVE_OUT}/threshold.json", threshold, binding)

## Evaluation: LOCKED until all detectors are frozen and the custodian releases the evaluation view

In [ ]:
RELEASE_EVALUATION = False
assert RELEASE_EVALUATION, "Evaluation is locked until all detectors are frozen"
from copy import deepcopy
from shared_protocol.evaluation import score_evaluation, evaluate, save_evaluation
eval_ids = set(binding.ids("evaluation"))
eval_records = [deepcopy(r) for r in manifest["adapters"] if r["adapter_id"] in eval_ids]
stage(eval_records)
scored = score_evaluation(detector, eval_records, binding, threshold)
save_evaluation(f"{DRIVE_OUT}/evaluation.json", binding, eval_records, scored, threshold)
report = evaluate(binding, eval_records, scored, threshold)
print(report["heldout_benign"]["pooled"]["fpr"], report["heldout_malicious"]["pooled"]["recall"])